In [6]:
!pip install -q torchinfo

In [7]:
import torch
import torch.nn as nn

# ── Standard autoencoder: encoder + bottleneck + decoder ─────────────
class Autoencoder(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        # Encoder: compresses 1×28×28 → latent_dim
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, 3, stride=2, padding=1),  # → 32×14×14
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), # → 64×7×7
            nn.ReLU(),
            nn.Flatten(),                               # → 64*7*7 = 3136
            nn.Linear(3136, latent_dim),               # → latent_dim
        )
        # Decoder: expands latent_dim → 1×28×28
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 3136),
            nn.ReLU(),
            nn.Unflatten(1, (64, 7, 7)),                # → 64×7×7
            nn.ConvTranspose2d(64, 32, 3, stride=2,    # → 32×14×14
                               padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 1, 3, stride=2,     # → 1×28×28
                               padding=1, output_padding=1),
            nn.Sigmoid(),   # pixel values in [0,1]
        )

    def encode(self, x):  return self.encoder(x)
    def decode(self, h):  return self.decoder(h)
    def forward(self, x): return self.decode(self.encode(x))

model = Autoencoder(latent_dim=32)
x = torch.randn(8, 1, 28, 28)   # batch of 8 MNIST images
x_hat = model(x)
print(f'Input:  {x.shape}')      # (8, 1, 28, 28)
print(f'Output: {x_hat.shape}')  # (8, 1, 28, 28)  — same shape

# Loss: MSE between reconstruction and original
loss = nn.MSELoss()(x_hat, x)
print(f'Reconstruction loss: {loss.item():.4f}')

# ── Training loop (unsupervised — no labels needed!) ──────────────────
optimiser = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(3):   # abbreviated for illustration
    optimiser.zero_grad()
    x_hat = model(x)
    loss  = nn.MSELoss()(x_hat, x)
    loss.backward()
    optimiser.step()
    print(f'Epoch {epoch+1}  loss: {loss.item():.4f}')

# ── Latent space interpolation ────────────────────────────────────────
model.eval()
with torch.no_grad():
    h1 = model.encode(x[[0]])   # latent vector for image 0
    h2 = model.encode(x[[1]])   # latent vector for image 1

    # Interpolate: α=0 → h1, α=1 → h2
    alphas = torch.linspace(0, 1, 8)
    for alpha in alphas:
        h_interp = (1 - alpha) * h1 + alpha * h2
        x_interp = model.decode(h_interp)   # decode the interpolated latent
        # In practice: visualise x_interp to see smooth transition


Input:  torch.Size([8, 1, 28, 28])
Output: torch.Size([8, 1, 28, 28])
Reconstruction loss: 1.2362
Epoch 1  loss: 1.2362
Epoch 2  loss: 1.2255
Epoch 3  loss: 1.2038


In [12]:
from torchinfo import summary
summary(model, input_size=(8, 1, 28, 28), col_width=20)

Layer (type:depth-idx)                   Output Shape         Param #
Autoencoder                              [8, 1, 28, 28]       --
├─Sequential: 1-1                        [8, 32]              --
│    └─Conv2d: 2-1                       [8, 32, 14, 14]      320
│    └─ReLU: 2-2                         [8, 32, 14, 14]      --
│    └─Conv2d: 2-3                       [8, 64, 7, 7]        18,496
│    └─ReLU: 2-4                         [8, 64, 7, 7]        --
│    └─Flatten: 2-5                      [8, 3136]            --
│    └─Linear: 2-6                       [8, 32]              100,384
├─Sequential: 1-2                        [8, 1, 28, 28]       --
│    └─Linear: 2-7                       [8, 3136]            103,488
│    └─ReLU: 2-8                         [8, 3136]            --
│    └─Unflatten: 2-9                    [8, 64, 7, 7]        --
│    └─ConvTranspose2d: 2-10             [8, 32, 14, 14]      18,464
│    └─ReLU: 2-11                        [8, 32, 14, 14]      --
│

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ── Denoising Autoencoder ─────────────────────────────────────────────
class DenoisingAE(nn.Module):
    def __init__(self, noise_std=0.2):
        super().__init__()
        self.noise_std = noise_std
        self.encoder = nn.Sequential(
            nn.Linear(784, 256), nn.ReLU(),
            nn.Linear(256,  64), nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear( 64, 256), nn.ReLU(),
            nn.Linear(256, 784), nn.Sigmoid(),
        )

    def corrupt(self, x):
        """Add Gaussian noise during training only."""
        if self.training:
            return x + self.noise_std * torch.randn_like(x)
        return x   # no corruption at inference

    def forward(self, x):
        x_noisy = self.corrupt(x).clamp(0, 1)   # add noise
        h       = self.encoder(x_noisy)          # encode noisy input
        x_hat   = self.decoder(h)                # decode to clean output
        return x_hat

# CRITICAL: Loss compares reconstruction with CLEAN input, not noisy input
dae = DenoisingAE(noise_std=0.3)
x_clean = torch.rand(16, 784)   # clean images
x_recon = dae(x_clean)          # internally corrupts, then reconstructs
loss    = nn.MSELoss()(x_recon, x_clean)  # compare with clean original
print(f'DAE reconstruction loss: {loss.item():.4f}')

# ── Sparse Autoencoder (sparsity via L1 penalty on activations) ───────
class SparseAE(nn.Module):
    def __init__(self, latent_dim=64, sparsity_weight=1e-3):
        super().__init__()
        self.sparsity_weight = sparsity_weight
        self.encoder = nn.Sequential(nn.Linear(784, latent_dim), nn.Sigmoid())
        self.decoder = nn.Sequential(nn.Linear(latent_dim, 784), nn.Sigmoid())

    def forward(self, x):
        h     = self.encoder(x)
        x_hat = self.decoder(h)
        # L1 penalty on activations encourages sparsity
        sparsity_loss = self.sparsity_weight * h.abs().mean()
        return x_hat, sparsity_loss

sparse_ae = SparseAE(latent_dim=128)
x = torch.rand(16, 784)
x_hat, sp_loss = sparse_ae(x)
recon_loss = nn.MSELoss()(x_hat, x)
total_loss = recon_loss + sp_loss
print(f'Recon: {recon_loss:.4f}  Sparsity: {sp_loss:.4f}  Total: {total_loss:.4f}')


DAE reconstruction loss: 0.0848
Recon: 0.0887  Sparsity: 0.0005  Total: 0.0892


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ════════════════════════════════════════════════════════════════════
# VARIATIONAL AUTOENCODER — side-by-side with standard AE
# The ONLY architectural difference is the encoder output and
# the reparameterisation trick. Everything else is identical.
# ════════════════════════════════════════════════════════════════════

class VAE(nn.Module):
    def __init__(self, latent_dim=20):
        super().__init__()
        self.latent_dim = latent_dim

        # Encoder: outputs TWO vectors (mu and log_var) instead of one
        self.encoder_conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, stride=2, padding=1), nn.ReLU(),  # 14×14
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(), # 7×7
            nn.Flatten()                                           # 3136
        )
        self.fc_mu      = nn.Linear(3136, latent_dim)   # mean
        self.fc_log_var = nn.Linear(3136, latent_dim)   # log(σ²)

        # Decoder: identical to standard AE decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 3136), nn.ReLU(),
            nn.Unflatten(1, (64, 7, 7)),
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 1,  3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid()
        )

    def encode(self, x):
        h       = self.encoder_conv(x)
        mu      = self.fc_mu(h)
        log_var = self.fc_log_var(h)
        return mu, log_var

    def reparameterise(self, mu, log_var):
        """Reparameterisation trick: z = mu + std * eps, eps ~ N(0,1)."""
        std = torch.exp(0.5 * log_var)   # sigma = exp(log_var / 2)
        eps = torch.randn_like(std)       # sample from N(0,1)
        return mu + std * eps             # differentiable w.r.t. mu, std

    def decode(self, z):  return self.decoder(z)

    def forward(self, x):
        mu, log_var = self.encode(x)
        z           = self.reparameterise(mu, log_var)  # stochastic
        x_hat       = self.decode(z)
        return x_hat, mu, log_var

def vae_loss(x_hat, x, mu, log_var, beta=1.0):
    """
    VAE loss = Reconstruction loss + beta * KL divergence.
    Reconstruction: how well does x_hat match x?
    KL:             how close is N(mu, sigma^2) to N(0,1)?
    """
    # Reconstruction loss (MSE or BCE)
    recon = F.mse_loss(x_hat, x, reduction='sum') / x.size(0)

    # KL divergence: (1/2) * sum(mu^2 + sigma^2 - log(sigma^2) - 1)
    # Using log_var = log(sigma^2) for numerical stability
    kl = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp()) / x.size(0)

    return recon + beta * kl, recon, kl

# ── Training step ─────────────────────────────────────────────────────
vae = VAE(latent_dim=20)
opt = torch.optim.Adam(vae.parameters(), lr=1e-3)

x = torch.rand(32, 1, 28, 28)   # batch of images
opt.zero_grad()
x_hat, mu, log_var = vae(x)
loss, recon, kl = vae_loss(x_hat, x, mu, log_var, beta=1.0)
loss.backward()
opt.step()
print(f'Total: {loss:.4f}  Recon: {recon:.4f}  KL: {kl:.4f}')

# ── Generating new images (no encoder needed!) ────────────────────────
vae.eval()
with torch.no_grad():
    # Sample from the prior N(0,1) — the KL term ensures this is valid
    z_random = torch.randn(16, 20)   # 16 random latent vectors
    generated = vae.decode(z_random)
    print(f'Generated images shape: {generated.shape}')   # (16, 1, 28, 28)

# ── Latent space interpolation ────────────────────────────────────────
with torch.no_grad():
    mu1, _ = vae.encode(x[[0]])   # mean encoding of image 0
    mu2, _ = vae.encode(x[[1]])   # mean encoding of image 1
    # Interpolate in latent space (use means, not samples, for clarity)
    alphas   = torch.linspace(0, 1, 8)
    z_interp = torch.stack([a * mu2 + (1-a) * mu1 for a in alphas])
    x_interp = vae.decode(z_interp.squeeze(1))
    print(f'Interpolated sequence shape: {x_interp.shape}')  # (8, 1, 28, 28)


Total: 68.5659  Recon: 68.5190  KL: 0.0469
Generated images shape: torch.Size([16, 1, 28, 28])
Interpolated sequence shape: torch.Size([8, 1, 28, 28])


In [4]:
import torch
import torch.nn as nn

# ════════════════════════════════════════════════════════════════════
# DCGAN: Deep Convolutional GAN
# Generator: noise → image (uses ConvTranspose2d for upsampling)
# Discriminator: image → real/fake probability (uses strided Conv2d)
# ════════════════════════════════════════════════════════════════════

class Generator(nn.Module):
    def __init__(self, z_dim=100, img_channels=1, features=64):
        super().__init__()
        # z_dim → 7×7×(features*2) → 14×14×features → 28×28×img_channels
        self.net = nn.Sequential(
            # Project noise to spatial feature map
            nn.ConvTranspose2d(z_dim, features*2, 7, 1, 0, bias=False),  # 7×7
            nn.BatchNorm2d(features*2), nn.ReLU(True),
            # Upsample to 14×14
            nn.ConvTranspose2d(features*2, features, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features), nn.ReLU(True),
            # Upsample to 28×28
            nn.ConvTranspose2d(features, img_channels, 4, 2, 1, bias=False),
            nn.Tanh()    # output in [-1, 1]; normalise real images to [-1,1] too
        )

    def forward(self, z):
        # z shape: (batch, z_dim, 1, 1) — treat noise as a 1×1 'feature map'
        return self.net(z)

class Discriminator(nn.Module):
    def __init__(self, img_channels=1, features=64):
        super().__init__()
        # 28×28 → 14×14 → 7×7 → scalar
        self.net = nn.Sequential(
            nn.Conv2d(img_channels, features,   4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),    # Leaky ReLU — standard for D
            nn.Conv2d(features,     features*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features*2), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(features*2,   1,          7, 1, 0, bias=False),
            nn.Sigmoid()   # D(x) = P(real)
        )

    def forward(self, x):  return self.net(x).view(-1)   # scalar per image

# ── Initialise weights with DCGAN paper convention ────────────────────
def weights_init(m):
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.normal_(m.weight, 0.0, 0.02)
    elif isinstance(m, nn.BatchNorm2d):
        nn.init.normal_(m.weight, 1.0, 0.02)
        nn.init.zeros_(m.bias)

G = Generator(z_dim=100).apply(weights_init)
D = Discriminator().apply(weights_init)

# ── CRITICAL: two SEPARATE optimisers ─────────────────────────────────
# G and D are updated independently — never update both at once
opt_D = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_G = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
criterion = nn.BCELoss()

# ── One training iteration ────────────────────────────────────────────
batch_size = 32
z_dim      = 100
real_images = torch.rand(batch_size, 1, 28, 28) * 2 - 1   # fake 'real' data

real_labels = torch.ones(batch_size)    # 1 = real
fake_labels = torch.zeros(batch_size)   # 0 = fake

# ── Step 1: Train Discriminator ───────────────────────────────────────
# 'Maximise log D(x) + log(1 - D(G(z)))'
opt_D.zero_grad()

# Loss on real images: D should output 1 for all of them
d_real = D(real_images)
loss_D_real = criterion(d_real, real_labels)

# Loss on fake images: D should output 0 for all of them
z = torch.randn(batch_size, z_dim, 1, 1)   # sample noise
fake_images = G(z).detach()                 # detach: don't update G here
d_fake = D(fake_images)
loss_D_fake = criterion(d_fake, fake_labels)

loss_D = loss_D_real + loss_D_fake
loss_D.backward()
opt_D.step()

# ── Step 2: Train Generator ───────────────────────────────────────────
# 'Minimise log(1 - D(G(z)))' ≡ 'Maximise log D(G(z))' (non-saturating)
opt_G.zero_grad()

z = torch.randn(batch_size, z_dim, 1, 1)
fake_images = G(z)          # generate again (this time we DO want gradients)
d_fake_for_G = D(fake_images)
# Use real_labels: G wants D to think fake images ARE real
loss_G = criterion(d_fake_for_G, real_labels)
loss_G.backward()
opt_G.step()

print(f'D loss: {loss_D.item():.4f}   G loss: {loss_G.item():.4f}')
# Healthy training: both losses should be around log(2) ≈ 0.69
# D >> G loss: D is dominating, G is getting no signal
# G >> D loss: G is dominating (or D is not training)

# ── Generating images at inference time ───────────────────────────────
G.eval()
with torch.no_grad():
    z = torch.randn(16, z_dim, 1, 1)
    generated = G(z)   # shape: (16, 1, 28, 28), values in [-1, 1]
    print(f'Generated: {generated.shape}')


D loss: 1.4011   G loss: 1.2290
Generated: torch.Size([16, 1, 28, 28])


In [5]:
import torch
import torch.nn as nn

# ── The three models share the same encoder/decoder backbone ──────────
# This makes the differences between them maximally clear.

def make_encoder(latent_dim):
    return nn.Sequential(
        nn.Linear(784, 256), nn.ReLU(),
        nn.Linear(256, latent_dim)
    )

def make_decoder(latent_dim):
    return nn.Sequential(
        nn.Linear(latent_dim, 256), nn.ReLU(),
        nn.Linear(256, 784), nn.Sigmoid()
    )

# ── STANDARD AE: encoder outputs one vector ───────────────────────────
class AE(nn.Module):
    def __init__(self, d=32):
        super().__init__()
        self.enc = make_encoder(d)
        self.dec = make_decoder(d)
    def forward(self, x):
        return self.dec(self.enc(x.flatten(1)))
    # Generation: NOT possible without manual sampling heuristics

# ── VAE: encoder outputs (mu, log_var); reparameterisation trick ──────
class VAESimple(nn.Module):
    def __init__(self, d=32):
        super().__init__()
        self.enc_shared = nn.Sequential(nn.Linear(784, 256), nn.ReLU())
        self.fc_mu      = nn.Linear(256, d)
        self.fc_lv      = nn.Linear(256, d)
        self.dec        = make_decoder(d)
    def forward(self, x):
        h   = self.enc_shared(x.flatten(1))
        mu  = self.fc_mu(h)
        lv  = self.fc_lv(h)
        std = torch.exp(0.5 * lv)
        z   = mu + std * torch.randn_like(std)   # reparameterisation
        return self.dec(z), mu, lv
    def generate(self, n):
        with torch.no_grad():
            z = torch.randn(n, self.fc_mu.out_features)
            return self.dec(z).view(n, 1, 28, 28)

# ── GAN: no encoder at all at inference time ──────────────────────────
class SimpleGenerator(nn.Module):
    def __init__(self, z=100, d=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z, 256), nn.ReLU(),
            nn.Linear(256, 784), nn.Sigmoid()
        )
    def forward(self, z): return self.net(z)
    def generate(self, n, z_dim=100):
        with torch.no_grad():
            z = torch.randn(n, z_dim)
            return self.net(z).view(n, 1, 28, 28)

# ── What sets them apart in a nutshell ───────────────────────────────
# AE:  train with MSE(x_hat, x)
#      latent space: unstructured — BAD for generation
#      generation:   not practical
#
# VAE: train with MSE(x_hat, x) + KL(N(mu,s²) || N(0,1))
#      latent space: structured, continuous — GOOD for generation
#      generation:   sample z~N(0,1), decode(z)  ← works well
#      weakness:     blurry outputs (MSE reconstruction loss)
#
# GAN: train G and D with adversarial losses (no reconstruction loss)
#      latent space: unstructured (no encoder needed at inference)
#      generation:   sample z~N(0,1), G(z)  ← very sharp outputs
#      weakness:     notoriously unstable training
